# FOMC Analysis

**Study Overview:**

This analysis examines how ATM straddles on 15 liquid sector ETFs behave around FOMC decision dates from 2020 to 2025, covering 49 Federal Reserve meetings including the two emergency cuts in March 2020. The goal is to understand whether implied volatility around FOMC events is systematically mispriced across different sectors of the market — and if so, whether a long or short straddle strategy would have been consistently profitable, and which sectors show the strongest and most reliable signals.

**Data**

Options data was collected using the Polygon.io historical API. For each of the 49 FOMC meetings and each of the 15 ETFs (SPY, QQQ, XLK, XLF, XLE, XLI, XLV, XLB, XLP, GLD, TLT, IWM, EEM, XOP, ITA), we fetched the at-the-money straddle price at every combination of entry timing (T-3, T-2, T-1, T=0 relative to the FOMC date) and exit timing (T=0, T+1, T+2, T+3). ATM strike selection used the globally nearest strike to the spot price, with all filters from Gao, Xing & Zhang (2018) applied at entry — minimum option price of $0.125, stock price above $5, days to expiry between 10 and 60, abs(delta) between 0.375 and 0.625, and moneyness between 0.9 and 1.1. Underlying spot prices were fetched from yfinance and reverse-adjusted for any stock splits to match Polygon's historical strike basis. The final clean dataset contains 4,900 rows across 15 tickers, 49 meetings, and 15 valid entry/exit combinations (T0/T0 excluded as it always yields zero).

**Analysis approach**

We analyse the data in three layers. First, at the combo level — a heatmap matrix showing average return and win rate for every entry/exit combination pooled across all 15 ETFs, to identify which timing windows have the most consistent edge. Second, at the ticker level — the same matrix broken down per ETF, to see whether certain sectors (e.g. TLT for rate-sensitive, XLE for energy, ITA for defense) behave differently around FOMC than the broad market SPY benchmark. Third, at the event level — year-by-year and meeting-by-meeting breakdowns to understand whether the signal is consistent across different rate regimes (2020 zero-rate era, 2022 aggressive hiking cycle, 2024 cutting cycle). The return metric throughout is pnl_pct — the percentage return on the straddle premium paid, assuming equal dollar sizing per position.

## Part 1: Data Loading & Processing

In [2]:
# Import neccessary Libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

In [3]:
# Load the dataset
df = pd.read_csv("options_data_fomc/fomc_etf_options.csv")
df["fomc_date"] = pd.to_datetime(df["fomc_date"])
df["entry_date"] = pd.to_datetime(df["entry_date"])
df["exit_date"]  = pd.to_datetime(df["exit_date"])

In [4]:
# Dedup - keep highest entry_straddle_mid per combo key
KEY = ["ticker","fomc_date","entry_offset","exit_offset"]
before = len(df)
df = (df.sort_values("entry_straddle_mid", ascending=False)
        .drop_duplicates(subset=KEY, keep="first")
        .reset_index(drop=True))
print(f"Dedup: {before:,} - {len(df):,} rows (removed {before-len(df):,})")

Dedup: 4,900 - 4,900 rows (removed 0)


In [5]:
# Evaluate Overview
print("Dataset Overview:")
print(f"  Rows: {len(df):,}")
print(f"  Tickers: {df['ticker'].nunique()}  {sorted(df['ticker'].unique())}")
print(f"  FOMC meetings: {df['fomc_date'].nunique()}")
print(f"  Date range: {df['fomc_date'].min().date()} → {df['fomc_date'].max().date()}")
print(f"  Years: {sorted(df['year'].unique())}")
print(f"  Entry offsets: {sorted(df['entry_offset'].unique())}")
print(f"  Exit offsets: {sorted(df['exit_offset'].unique())}")
print(f"  Combos: {df.groupby(['entry_offset','exit_offset']).ngroups}")

Dataset Overview:
  Rows: 4,900
  Tickers: 15  ['EEM', 'GLD', 'ITA', 'IWM', 'QQQ', 'SPY', 'TLT', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLV', 'XOP']
  FOMC meetings: 49
  Date range: 2020-01-29 → 2025-12-10
  Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Entry offsets: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(0)]
  Exit offsets: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
  Combos: 16


In [6]:
# Evaluate rows per ticker with coverage metric
TOTAL_MEETINGS  = 49
VALID_COMBOS    = 15   # 16 combos minus T0/T0
EXPECTED_PER_TICKER = TOTAL_MEETINGS * VALID_COMBOS  # 735 max

print("Rows per ticker (with coverage):")
ticker_summary = (df.groupby("ticker")
                    .agg(
                        n            = ("pnl_pct",    "count"),
                        meetings     = ("fomc_date",  "nunique"),
                        avg_ret      = ("pnl_pct",    "mean"),
                        win_rate     = ("pnl_pct",    lambda x: (x>0).mean()),
                        avg_entry_iv = ("entry_atm_iv","mean"),
                    )
                    .round(4)
                    .sort_values("n", ascending=False))

# Coverage = rows collected / max possible rows
ticker_summary["expected"]     = EXPECTED_PER_TICKER
ticker_summary["coverage_pct"] = (ticker_summary["n"] / EXPECTED_PER_TICKER * 100).round(1)

# Meeting coverage = meetings seen / total FOMC meetings
ticker_summary["meeting_cov_pct"] = (ticker_summary["meetings"] / TOTAL_MEETINGS * 100).round(1)

# Reorder columns for readability
ticker_summary = ticker_summary[[
    "n", "expected", "coverage_pct",
    "meetings", "meeting_cov_pct",
    "avg_ret", "win_rate", "avg_entry_iv"
]]

ticker_summary.columns = [
    "rows", "expected", "coverage_%",
    "meetings", "meeting_cov_%",
    "avg_ret", "win_rate", "avg_iv"
]

print(ticker_summary.to_string())
print(f"\nOverall coverage: {len(df):,} / {EXPECTED_PER_TICKER * df['ticker'].nunique():,} "
      f"({len(df) / (EXPECTED_PER_TICKER * df['ticker'].nunique()) * 100:.1f}%)")

Rows per ticker (with coverage):
        rows  expected  coverage_%  meetings  meeting_cov_%  avg_ret  win_rate  avg_iv
ticker                                                                                
QQQ      703       735        95.6        49          100.0   0.0059    0.3670  0.2854
GLD      700       735        95.2        48           98.0  -0.0265    0.3329  0.1674
SPY      601       735        81.8        48           98.0  -0.0400    0.3827  0.3238
IWM      598       735        81.4        48           98.0  -0.0413    0.3127  0.3698
XLF      465       735        63.3        47           95.9   0.0010    0.3806  0.3405
XLK      438       735        59.6        46           93.9   0.0312    0.4064  0.2820
XLE      276       735        37.6        39           79.6  -0.0772    0.3188  0.5267
XOP      198       735        26.9        40           81.6  -0.0799    0.2879  0.6808
XLI      188       735        25.6        35           71.4  -0.0613    0.3085  0.2788
EEM      1

In [7]:
# Check missing data
print("Evaluate Missing Data:D")
key_cols = ["pnl_pct","entry_straddle_mid","exit_straddle_mid",
            "entry_atm_iv","exit_atm_iv","stock_move"]
print(df[key_cols].isnull().sum().to_string())

Evaluate Missing Data:D
pnl_pct                0
entry_straddle_mid     0
exit_straddle_mid      0
entry_atm_iv           0
exit_atm_iv           70
stock_move             0


In [8]:
# Sample rows 
print("Sample Rows (5 random):")
display_cols = ["ticker","fomc_date","entry_offset","exit_offset",
                "entry_straddle_mid","exit_straddle_mid",
                "pnl_pct","entry_atm_iv","exit_atm_iv","stock_move"]
print(df[display_cols].sample(5, random_state=42).to_string(index=False))

Sample Rows (5 random):
ticker  fomc_date  entry_offset  exit_offset  entry_straddle_mid  exit_straddle_mid   pnl_pct  entry_atm_iv  exit_atm_iv  stock_move
   EEM 2025-12-10            -3            3              1.5710             1.0441 -0.335391      0.186332     0.199415   -0.012044
   GLD 2022-09-21            -3            3              4.3200             4.1042 -0.049954      0.177488     0.264812   -0.029582
   QQQ 2022-12-14            -3            1             14.3420             5.8793 -0.590064      0.393230     0.452877   -0.018236
   TLT 2025-09-17            -2            1              2.3457             1.4487 -0.382402      0.190657     0.288589   -0.010727
   XLI 2025-07-30             0            1              3.7815             3.9953  0.056538      0.150073     0.163765    0.000066


## Part 2: Configuration Analysis

With clean data confirmed across all 15 ETFs and 49 FOMC meetings, we now analyse how straddle returns vary across every entry/exit timing combination. The goal is to identify which configurations — defined by how many days before or after the FOMC decision you enter and exit, produce the most consistent edge when pooled across the entire ETF universe. This gives us a baseline view of the FOMC straddle timing structure before breaking it down by individual ticker.

In [9]:
# Import neccessary libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [10]:
# Set up NUSA standard visualization template
BG    = "#ffffff"
PANEL = "#f8f8f8"
RED   = "#e8133a"
GREEN = "#2db82d"
GREY  = "#555555"
DG    = "#dddddd"
FONT  = "Garamond, 'EB Garamond', Georgia, serif"

In [11]:
# Set e_offs and x_offs for consistent ordering
e_offs = [-3, -2, -1, 0]
x_offs = [0, 1, 2, 3]
df = df[~((df["entry_offset"]==0) & (df["exit_offset"]==0))].copy()

In [12]:
# Set up Ticker Configurations
TICKER_META = {
    "SPY": "S&P 500",
    "QQQ": "Nasdaq 100",
    "XLK": "Technology",
    "XLF": "Financials",
    "XLE": "Energy",
    "XLI": "Industrials",
    "XLV": "Health Care",
    "XLB": "Materials",
    "XLP": "Consumer Staples",
    "GLD": "Gold",
    "TLT": "Long Treasuries",
    "IWM": "Russell 2000",
    "EEM": "Emerging Markets",
    "XOP": "Oil & Gas E&P",
    "ITA": "Aerospace & Defense",
}

In [13]:
# Function to plot heatmap for a given ticker
def plot_ticker(ticker):
    sub  = df[df["ticker"] == ticker]
    name = TICKER_META.get(ticker, ticker)

    st = (sub.groupby(["entry_offset","exit_offset"])
             .agg(
                 n        = ("pnl_pct","count"),
                 avg_ret  = ("pnl_pct","mean"),
                 win_rate = ("pnl_pct", lambda x: (x>0).mean()),
             )
             .reset_index())
    idx_t = st.set_index(["entry_offset","exit_offset"])

    z, txt = [], []
    for e in e_offs:
        rz, rt = [], []
        for x in x_offs:
            if e == 0 and x == 0:
                rz.append(None); rt.append("excl.")
            elif (e, x) in idx_t.index:
                v  = idx_t.loc[(e,x), "avg_ret"]
                wr = idx_t.loc[(e,x), "win_rate"]
                n  = int(idx_t.loc[(e,x), "n"])
                rz.append(v)
                rt.append(f"{v:.2%}\nWR {wr:.0%}  n={n}")
            else:
                rz.append(None); rt.append("")
        z.append(rz); txt.append(rt)

    fig = go.Figure(go.Heatmap(
        z=z,
        x=[f"Exit T{x:+d}" for x in x_offs],
        y=[f"Entry T{e:+d}" for e in e_offs],
        text=txt, texttemplate="%{text}",
        textfont=dict(family=FONT, size=11, color="#111111"),
        colorscale=[[0,RED],[0.45,"#f5a0aa"],[0.5,"#ffffff"],
                    [0.55,"#a0e6a0"],[1,GREEN]],
        zmid=0, showscale=True,
        colorbar=dict(tickformat=".0%",
                      tickfont=dict(family=FONT, color=GREY, size=11),
                      outlinecolor=DG, outlinewidth=1),
        hoverongaps=False,
    ))
    fig.update_layout(
        title=dict(text=f"{ticker}  ·  {name}  -  Avg Return per Combo",
                   font=dict(family=FONT, size=19, color="#111111"),
                   x=0.03, xanchor="left"),
        paper_bgcolor=BG, plot_bgcolor=PANEL,
        font=dict(family=FONT, color="#111111", size=13),
        height=380, margin=dict(l=65, r=40, t=65, b=40),
        hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                        font=dict(family=FONT, color="#111111", size=13)),
    )
    fig.update_xaxes(side="top", tickfont=dict(family=FONT, color="#111111", size=12),
                     gridcolor=DG, linecolor=DG)
    fig.update_yaxes(autorange="reversed",
                     tickfont=dict(family=FONT, color="#111111", size=12),
                     gridcolor=DG, linecolor=DG)
    fig.show()

In [14]:
# Analyse SPY configs
plot_ticker("SPY")

**Analysis:** 

The S&P 500 straddle around FOMC meetings shows a clear and consistent pattern where the only configurations with positive average returns are short-duration holds that exit on the decision day itself (T+0). Entering at T-2 and exiting at T+0 gives the best result at +2.41% with a 49% win rate, followed by E-1/X+0 at +1.81% with a 50% win rate. Both barely above breakeven on win rate, which tells you the edge here is not from winning more often but from the occasional large gain when the market moves sharply on the Fed decision. The moment you hold beyond T+0, returns deteriorate sharply and consistently. By T+2 and T+3, nearly every entry combination is producing losses of 5–16%, with win rates collapsing to the mid-30s. 

This is indicative of a classic IV crush, where once the decision is announced, the uncertainty premium that was holding up straddle prices evaporates within hours, and holding through the next day or two means you are fighting aggressive theta decay on top of collapsing implied volatility. The E-3/X+2 combination at -16.46% is the worst of the lot, showing what happens when you enter early to capture pre-meeting IV run-up but then overstay your welcome by two days post-announcement. The takeaway for SPY specifically is that the FOMC straddle is a very short-duration trade - enter 1-2 days before, exit same day as the decision, and get out before the crush begins. Alternatively, one could short the straddle for certain configurations, betting that SPY stays within a specific range. Combining the long-short strategy based on these configurations can result in potentially better risk adjusted returns. 

In [15]:
# Analyse QQQ (Nasdaq 100) configs
plot_ticker("QQQ")

**Analysis:**

QQQ tells a meaningfully different story from SPY. Unlike the broad market, the Nasdaq 100 straddle shows positive returns not just at T+0 exit but consistently through T+1 and T+2 as well, with E-1/X+1 being the standout combo at +3.71%, E-1/X+2 at +2.83%, and even E-1/X+3 remaining positive at +2.56%. This suggests that tech-heavy QQQ continues to move after the FOMC announcement rather than immediately reverting, giving the straddle holder additional time to profit. The E-2 entry row is also broadly positive across T+0, T+1, and T+2 exits, reinforcing that entering 2 days before the meeting and holding through the next day or two has been a reliable trade on QQQ specifically. The key structural difference from SPY is that IV crush appears slower or less severe for QQQ, likely because rate sensitivity in mega-cap tech creates sustained directional movement post-announcement rather than a sharp one-day reaction followed by flat trading. The only clear losers are the T+0 exit combos for T-3 and T-1 entries (negative) and the T+3 exits for later entries, suggesting the edge does eventually get eaten by decay but takes longer than it does for SPY. Overall, QQQ supports a longer hold window than SPY around FOMC, with the E-1/X+1 combination standing out as the most compelling configuration.

In [16]:
# Analyse XLK (Technology) configs
plot_ticker("XLK")

**Analysis:**

XLK is the strongest result in the dataset so far and stands out from both SPY and QQQ in a striking way - the entire matrix is almost uniformly green, with returns that are substantially larger in magnitude than anything seen on the broader indices. The best combo is E+0/X+3 at +12.32% with a 44% win rate, followed by E+0/X+2 at +9.30% with a 58% win rate, suggesting that entering on the FOMC decision day itself and holding for 2-3 days has been exceptionally profitable on the technology sector ETF. What makes this particularly notable is that the E-2/X+0 and E-1/X+0 combos are also strongly positive at +7.19% and +6.49% respectively, with win rates of 53% and 60% - these are the highest win rates seen across any ticker so far, meaning XLK is not just generating large average returns from occasional blowouts but actually winning more often than it loses on those short-duration pre-announcement combos. The only red cells are E-3/X+1 at -3.70% and E-2/X+2 and E-2/X+3 which are mildly negative, suggesting that entering very early and holding through the post-announcement period is the one combination to avoid. The E+0 entry row being the strongest across all exit windows is a meaningful signal - it implies that on FOMC day itself, tech sector volatility is still underpriced even after the announcement, and the subsequent 2-3 days of continued sector rotation and rate repricing generate enough movement to profit. XLK warrants further investigation as the top-performing ticker in this dataset.

In [17]:
# Analyse XLF (Financial) configs
plot_ticker("XLF")

**Analysis:**

XLF is the most rate-sensitive equity sector and its pattern reflects that directly, though in a more nuanced way than expected. The strongest combo is E-3/X+2 at +7.30% with a 50% win rate, and E-2/X+0 at +5.01% with a 52% win rate and E-1/X+0 at +4.12% with the highest win rate in this matrix at 62%. The latter is particularly notable since it means entering one day before the Fed decision and exiting same day has won more often than not across 34 observations. The structural pattern here is distinct from both SPY and QQQ: XLF shows positive returns at T+0 and T+1 exits for early entries, but then flips sharply negative at T+2 and T+3. The E-2/X+3 combo at -6.44% and E-3/X+3 at -1.42% show that overstaying the trade in financials is especially punishing. This makes sense mechanically: bank stocks and financial ETFs reprice rapidly and decisively on the rate decision, capturing the move in full within the first day or two, after which mean reversion or sector rotation drags the straddle value down. The E-1/X+0 combo standing out with a 62% win rate is the cleanest signal in the entire XLF matrix, suggesting that the day-before-to-decision-day window is where financial sector volatility is most systematically underpriced heading into FOMC. The T+2 exit for the E-3 entry is an outlier worth flagging since the +7.30% return with 50% win rate suggests that when you enter very early and hold through two days post-announcement, you occasionally catch a large prolonged move in financials, but the distribution is likely fat-tailed rather than consistent.

In [18]:
# Analyse XLE (Energy) configs
plot_ticker("XLE")

**Analysis:**

XLE is the starkest result in the dataset so far, and the message is unambiguous: the only profitable configuration is exiting on the FOMC decision day itself, and everything beyond that is deeply negative. E-3/X+0 at +2.77% with a 68% win rate and E-2/X+0 at +2.05% with a 57% win rate are the only green cells in the entire matrix. The moment you hold past T+0, returns collapse dramatically. E-1/X+2 sits at -14.85% with a 19% win rate, E-1/X+3 at -20.24% with a 20% win rate, and E-2/X+3 at -19.14% with a 33% win rate. These are not just negative average returns but catastrophically low win rates suggesting near-consistent losses. The E-3/X+0 win rate of 68% is actually the highest single-cell win rate seen across all tickers analysed so far, which makes the T+0 exit signal for XLE particularly clean and reliable. The interpretation is straightforward: energy sector volatility spikes in anticipation of the rate decision, the straddle appreciates as IV rises into the announcement, and then the combination of IV crush and the fact that energy prices are driven by supply and demand rather than rate decisions means the post-FOMC drift provides no support at all. XLE is effectively a pure IV run-up trade with a very hard exit rule: you must be out by close on decision day, no exceptions.

In [19]:
# Analyse XLI (Financial) configs
plot_ticker("XLI")

**Analysis:**

XLI is one of the weakest results in the dataset and the coverage issue is immediately visible, sample sizes are as low as 7 and 8 for some cells, which means there is relatively less liquidity for the index and the numbers need to be interpreted with caution. That said, the directional picture is almost entirely negative. Every E-3 and E-2 entry combination is deeply in the red regardless of exit timing, with E-2/X+3 at -20.90% with a 0% win rate and E-3/X+3 at -20.36% with an 11% win rate being the worst readings seen across any ticker so far. The only cells approaching neutral or positive are in the E-1 and E+0 entry rows: E-1/X+2 at +0.64% with a 50% win rate and E+0/X+2 at +0.68% with a 56% win rate are the lone bright spots, though at n=10 and n=18 respectively these are too thin to draw strong conclusions from. The E-2/X+3 combination having a 0% win rate across 8 observations is a striking data point even with small sample size. The broad takeaway is that industrials do not have a meaningful relationship with FOMC events in terms of options pricing edge. Rate decisions affect capital costs and infrastructure spending indirectly rather than immediately, so there is no sharp volatility event to capture. The low coverage across this ticker also suggests XLI options frequently failed the liquidity and delta filters, which is itself a signal that the options market for this ETF is less active around FOMC dates.

In [20]:
# Analyse XLV (Healthcare) configs
plot_ticker("XLV")

**Analysis:**

XLV is the clearest non-event in the dataset. The entire matrix is red with no meaningful positive returns anywhere, and the coverage is among the lowest seen so far with several cells having n=6 to n=13. E-3/X+3 at -23.92% with a 25% win rate and E-3/X+1 at -9.10% with a 25% win rate reflect consistent losses rather than just bad luck on thin samples. The only cell even approaching positive is E+0/X+2 at +0.45% with a 57% win rate, but at n=7 this is too small to mean anything. The E+0/X+1 cell at -13.06% with a 0% win rate across 2 observations is too thin to interpret. The interpretation here is straightforward: healthcare is structurally disconnected from monetary policy. Drug approvals, Medicare pricing, clinical trial outcomes and sector regulation are what move XLV, none of which are influenced by whether the Fed hikes 25 or 50 basis points. There is no pre-FOMC IV run-up to capture because the options market correctly recognises that FOMC is not a meaningful catalyst for this sector. The poor coverage also reinforces this point - XLV options around FOMC dates frequently did not meet the liquidity and delta filters, suggesting market makers are not actively building IV into XLV options ahead of Fed meetings the way they are for rate-sensitive sectors like XLF or macro-sensitive ones like QQQ and XLK. XLV should be excluded from any FOMC straddle strategy.

In [21]:
# Analyse XLB (Materials) configs
plot_ticker("XLB")

**Analysis:**

XLB presents a mixed and somewhat noisy picture, with coverage among the lowest in the dataset — several cells have n=4 to n=9, which makes any individual reading unreliable. That caveat stated, there are a few directional signals worth noting. The E-1 entry row is the strongest, with E-1/X+0 at +8.85% with a 50% win rate and E-1/X+1 at +2.63% with an 80% win rate. The 80% win rate on E-1/X+1 is the highest single-cell win rate seen across the entire study, but at n=5 it is too thin to be actionable on its own. E-2/X+1 at +7.06% with a 40% win rate is another positive data point, though again at n=5. The pattern of losses is concentrated in T+3 exits across all entry rows, with E-3/X+3 at -18.50% with a 0% win rate and E-2/X+3 at -13.18% with a 25% win rate being particularly severe. This aligns with what we have seen across most tickers: holding a straddle to T+3 after FOMC is almost universally a losing proposition. Materials sector sensitivity to FOMC is indirect, coming through commodity pricing, dollar strength, and credit conditions rather than a direct earnings catalyst, which may explain the inconsistency across configurations. Given the thin coverage and mixed signals, XLB does not provide a clean tradeable pattern and warrants caution before drawing conclusions. More data points would be needed to validate any of the positive cells here.

In [22]:
# Analyse XLP (Consumer Staples) configs
plot_ticker("XLP")

**Analysis:**

XLP is largely uninvestable from a FOMC straddle perspective, and the coverage numbers tell most of the story before even looking at the returns. With sample sizes as low as n=3 and n=4 across multiple cells, almost nothing here is statistically meaningful. What is visible directionally is mostly negative, with the E-3 row being uniformly and severely red across all exits - E-3/X+0 at -12.64% with a 0% win rate and E-3/X+2 at -16.04% with a 0% win rate are among the worst raw numbers in the entire study, though at n=4 and n=6 respectively these could easily be driven by one or two catastrophic observations. The only positive cells are E-2/X+2 at +3.26% with a 57% win rate at n=7, E-1/X+3 at +7.60% with a 62% win rate at n=8, and E+0/X+1 at +0.85% with a 67% win rate at n=3. The E-1/X+3 result is superficially interesting but at n=8 with a 62% win rate it could simply reflect a few large post-FOMC moves during the 2022 rate shock period that happened to benefit the straddle. Consumer staples are the quintessential defensive sector with very low beta and limited sensitivity to rate decisions in either direction, which explains why XLP options markets do not build meaningful IV into FOMC dates. Like XLV, this ticker should be excluded from any systematic FOMC straddle strategy, and the thin coverage reinforces that conclusion


In [23]:
# Analyse GLD (Gold) configs
plot_ticker("GLD")

**Analysis:**

GLD presents a fascinating and structurally distinct pattern from all equity ETFs analysed so far. The positive returns are concentrated entirely in the E+0 and E-1 entry rows, and specifically in the T+1 and T+2 exit windows rather than T+0. E-1/X+1 at +5.38% with a 48% win rate and E+0/X+2 at +4.49% with a 49% win rate are the standout combos, with E+0/X+1 at +1.18% and E+0/X+3 at +0.52% also slightly positive. This tells a fundamentally different story from equity sectors: gold does not react sharply on the FOMC announcement day itself but continues moving in the day or two following the decision. This makes intuitive sense given gold's relationship with real interest rates and the dollar. The initial market reaction to a rate decision often takes a day or two to fully transmit into gold prices as traders reassess inflation expectations, dollar strength, and safe haven demand in light of the new policy signal. Entering on decision day and holding through T+1 or T+2 captures this delayed repricing. What is equally notable is the consistent underperformance of the E-3 row across all exits, with returns of -4.22% to -16.29% and win rates as low as 24%. Entering gold straddles three days before FOMC and holding through any exit window has been a reliable way to lose money. The coverage on GLD is among the best in the dataset with n=40 to n=46 across most cells, giving this pattern considerably more statistical weight than the thin-coverage tickers seen earlier.

In [24]:
# Analyse TLT (Treasuries) configs
plot_ticker("TLT")

**Analysis:**

TLT is the most theoretically interesting ticker in the entire study given it is the most directly rate-sensitive instrument in the universe, and the results are both striking and paradoxical. The only positive configurations are the two shortest-duration pre-announcement entries that exit on decision day itself: E-3/X+0 at +2.78% with a 67% win rate and E-1/X+0 at +1.95% with a 50% win rate. Everything beyond T+0 exit is deeply negative, and the severity increases dramatically as the hold extends. E-2/X+3 at -32.75% with a 0% win rate and E-3/X+3 at -32.48% with a 33% win rate are the single worst readings in the entire dataset across any ticker or configuration. The T+3 column is catastrophic across all entry rows, with every cell losing between 9% and 33%. Coverage is thin at n=5 to n=8 throughout, so individual cells should be treated cautiously, but the directional consistency is hard to dismiss entirely. The paradox here is that TLT should theoretically be the best FOMC straddle candidate since it is most directly moved by rate decisions, yet it is one of the worst performers beyond T+0. The explanation is likely that TLT options are already pricing in substantial rate move uncertainty well before the meeting, meaning the IV run-up to T+0 is modest and the IV crush after is severe. Once the rate decision is known, treasury bond prices reprice immediately and completely, leaving nothing left for the straddle to capture on subsequent days. The near-zero win rates at T+3 suggest that TLT straddles held past the announcement date are losing on virtually every single meeting, pointing to an aggressive and systematic post-announcement IV crush in the treasury options market.

In [25]:
# Analyse IWM (Russell 2000) configs
plot_ticker("IWM")

**Analysis**

IWM is consistently negative across almost the entire matrix with one minor exception, and the pattern aligns closely with what we saw for SPY but with worse returns across the board. The only positive cell is E-1/X+0 at +1.06% with a 45% win rate, and even that is marginal. Every other configuration produces losses, with the E+0 entry row being particularly weak — E+0/X+1 at -6.19%, E+0/X+2 at -4.51%, and E+0/X+3 at -7.93% with a 21% win rate. The E-3/X+2 cell at -12.07% with a 33% win rate stands out as the worst single configuration. Coverage is reasonable at n=33 to n=42 for most cells, giving this picture more statistical weight than the thin-coverage tickers. What makes IWM's underperformance relative to SPY interesting is that small caps are theoretically more sensitive to domestic monetary policy than large caps, since smaller companies rely more on floating rate debt and are more exposed to credit conditions. Yet that sensitivity does not translate into exploitable straddle profits — if anything it works against the long straddle holder, suggesting that small cap volatility is already well-priced or even overpriced heading into FOMC meetings. The win rates hovering around 30-35% across most cells reinforce this: IWM straddles are losing more than two-thirds of the time across nearly every timing configuration, which is more consistent with a systematic overpricing of IWM volatility around Fed meetings than with an underpriceduncertainty story.

In [26]:
# Analyse EEM (Emerging Markets) configs
plot_ticker("EEM")

In [27]:
# Analyse XOP (Oil & Gas E&P) configs
plot_ticker("XOP")

In [28]:
# Analyse ITA (Aerospace & Defense) configs
plot_ticker("ITA")

**Interpretation & Next Steps**

Using the configuration analysis from Part 2 as a foundation, we now build and validate a systematic long/short straddle strategy using a walk-forward methodology to avoid look-ahead bias. The training period spans 2020 to 2024, during which we identify signal configurations with specific ticker and entry/exit offset combinations that meet strict criteria on both sides. Long signals require n > 20 observations, avg return above 2%, and a positive median return to confirm the edge is consistent rather than driven by outliers. Short signals require n > 20 observations, avg return below -2%, and a negative median return, meaning the long straddle loses consistently enough that selling the straddle would have been profitable. Once these signals are locked in from the training data without any further adjustment, we apply them directly to 2025 data as a true out-of-sample test. Short positions flip the sign of pnl_pct since we are now selling the straddle and collecting premium. The comparison table at the end of the output is the key deliverable as it shows side by side what each signal produced in training versus what it actually produced in 2025, giving us a direct read on which patterns were durable and which were specific to the 2020-2024 regime.


In [29]:
# Set Configs (Filters to take out any edge cases)
LONG_AVG_RET  =  0.02   # avg_ret > 2%
LONG_MED_RET  =  0.00   # median_ret > 0%
LONG_MIN_N    =  15     # n > 20

SHORT_AVG_RET = -0.02   # avg_ret < -2%
SHORT_MED_RET =  0.00   # median_ret < 0%
SHORT_MIN_N   =  15    # n > 20

In [30]:
train = df[df["year"] <= 2022].copy()
test  = df[df["year"] == 2023].copy()

print(f"Training set : {len(train):,} rows | {train['year'].nunique()} years (2020-2022)")
print(f"Test set     : {len(test):,}  rows | 2023 only")

Training set : 1,830 rows | 3 years (2020-2022)
Test set     : 675  rows | 2023 only


In [31]:
# Identify signals on training data (what configs to go long on, and what to go short on)
combo_stats = (train.groupby(["ticker","entry_offset","exit_offset"])
                    .agg(
                        n        = ("pnl_pct","count"),
                        avg_ret  = ("pnl_pct","mean"),
                        med_ret  = ("pnl_pct","median"),
                        win_rate = ("pnl_pct", lambda x: (x>0).mean()),
                        std      = ("pnl_pct","std"),
                    )
                    .reset_index())

long_signals = combo_stats[
    (combo_stats["n"]       > LONG_MIN_N)  &
    (combo_stats["avg_ret"] > LONG_AVG_RET) &
    (combo_stats["med_ret"] > LONG_MED_RET)
].copy()
long_signals["side"] = "LONG"

short_signals = combo_stats[
    (combo_stats["n"]       > SHORT_MIN_N)   &
    (combo_stats["avg_ret"] < SHORT_AVG_RET) &
    (combo_stats["med_ret"] < SHORT_MED_RET)
].copy()
short_signals["side"] = "SHORT"

print(f"\nSignals identified from training period (2020-2024):")
print(f"  Long  signals : {len(long_signals)}")
print(f"  Short signals : {len(short_signals)}")

print(f"\nLong signals:")
print(long_signals[["ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret", ascending=False)
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))

print(f"\nShort signals:")
print(short_signals[["ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret")
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))



Signals identified from training period (2020-2024):
  Long  signals : 2
  Short signals : 33

Long signals:
ticker  entry_offset  exit_offset  n avg_ret med_ret win_rate
   GLD            -1            1 23   7.74%   3.12%      57%
   GLD             0            2 22   3.47%   1.84%      55%

Short signals:
ticker  entry_offset  exit_offset  n avg_ret med_ret win_rate
   SPY            -1            3 16 -24.55% -18.91%      19%
   IWM            -1            3 17 -24.35% -34.72%      12%
   SPY            -1            2 19 -16.41% -19.66%      37%
   SPY            -2            2 16 -16.15% -17.49%      19%
   SPY            -2            3 17 -16.08% -23.79%      18%
   SPY             0            3 18 -15.74% -14.64%      28%
   IWM            -1            2 20 -13.66% -21.51%      20%
   IWM            -2            3 19 -12.96% -24.21%      26%
   SPY            -1            1 16 -12.57%  -9.00%      31%
   GLD            -3            3 22 -12.35% -14.95%      27%
   IWM

In [32]:
# Apply signals to test set and evaluate performance
long_keys  = set(zip(long_signals["ticker"],
                     long_signals["entry_offset"],
                     long_signals["exit_offset"]))
short_keys = set(zip(short_signals["ticker"],
                     short_signals["entry_offset"],
                     short_signals["exit_offset"]))

test = test.copy()
test["signal"] = None
test.loc[test.apply(lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"])
                    in long_keys,  axis=1), "signal"] = "LONG"
test.loc[test.apply(lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"])
                    in short_keys, axis=1), "signal"] = "SHORT"

test_signals = test[test["signal"].notna()].copy()

# Short side: flip pnl_pct
test_signals["strategy_pnl"] = test_signals.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1
)

print(f"\nTest trades: {len(test_signals)}")
print(f"  Long  trades : {(test_signals['signal']=='LONG').sum()}")
print(f"  Short trades : {(test_signals['signal']=='SHORT').sum()}")


Test trades: 248
  Long  trades : 15
  Short trades : 233


In [33]:
# Performance per ticker
print(f"\nPerformance by Ticker:")
ticker_perf = (test_signals.groupby(["ticker","signal"])
               .agg(
                   n        = ("strategy_pnl","count"),
                   avg_ret  = ("strategy_pnl","mean"),
                   win_rate = ("strategy_pnl", lambda x: (x>0).mean()),
                   total    = ("strategy_pnl","sum"),
               )
               .round(4))
print(ticker_perf.to_string(formatters={
    "avg_ret":  "{:.2%}".format,
    "win_rate": "{:.0%}".format,
    "total":    "{:.2f}".format,
}))


Performance by Ticker:
                n avg_ret win_rate  total
ticker signal                            
GLD    LONG    15   9.96%      40%   1.49
       SHORT   54   9.60%      72%   5.19
IWM    SHORT   56 -16.36%      55%  -9.16
QQQ    SHORT   65 -18.87%      45% -12.27
SPY    SHORT   58 -11.02%      36%  -6.39


In [34]:
print(f"\nTraining (2020-2024) vs Test (2025) — Signal Combos:")
print(f"{'Ticker':<6} {'E':>3} {'X':>3} {'Side':<6} "
      f"{'Train Avg':>10} {'Train WR':>9} {'Train N':>8} "
      f"{'Test Avg':>9} {'Test WR':>8} {'Test N':>7}")

all_signals = pd.concat([long_signals, short_signals])
for _, row in all_signals.sort_values(["side","ticker"]).iterrows():
    t, e, x, side = row["ticker"], int(row["entry_offset"]), int(row["exit_offset"]), row["side"]
    test_sub = test_signals[(test_signals["ticker"]==t) &
                             (test_signals["entry_offset"]==e) &
                             (test_signals["exit_offset"]==x)]
    if len(test_sub) == 0:
        test_avg, test_wr, test_n = float("nan"), float("nan"), 0
    else:
        pnl = test_sub["strategy_pnl"]
        test_avg = pnl.mean(); test_wr = (pnl>0).mean(); test_n = len(pnl)

    print(f"{t:<6} {e:>+3} {x:>+3} {side:<6} "
          f"{row['avg_ret']:>10.2%} {row['win_rate']:>9.0%} {int(row['n']):>8} "
          f"{test_avg:>9.2%} {test_wr:>8.0%} {test_n:>7}")


Training (2020-2024) vs Test (2025) — Signal Combos:
Ticker   E   X Side    Train Avg  Train WR  Train N  Test Avg  Test WR  Test N
GLD     -1  +1 LONG        7.74%       57%       23    11.77%      38%       8
GLD     +0  +2 LONG        3.47%       55%       22     7.90%      43%       7
GLD     -3  +0 SHORT      -4.58%       30%       23    10.50%      75%       8
GLD     -3  +2 SHORT      -7.95%       26%       23    20.40%      88%       8
GLD     -3  +3 SHORT     -12.35%       27%       22    18.01%      86%       7
GLD     -2  +0 SHORT      -2.73%       42%       24     3.89%      75%       8
GLD     -2  +2 SHORT      -3.84%       29%       24    10.01%      75%       8
GLD     -2  +3 SHORT      -8.70%       30%       23     3.44%      57%       7
GLD     -1  +0 SHORT      -2.16%       36%       22     1.26%      50%       8
IWM     -3  +1 SHORT      -7.80%       38%       16     9.37%      83%       6
IWM     -2  +1 SHORT      -6.69%       38%       16   -18.23%      62%       

In [37]:
all_periods = df.copy()
all_periods["signal"] = None
all_periods.loc[all_periods.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in long_keys, axis=1),
    "signal"] = "LONG"
all_periods.loc[all_periods.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in short_keys, axis=1),
    "signal"] = "SHORT"

strat = all_periods[all_periods["signal"].notna()].copy()
strat["strategy_pnl"] = strat.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1)
strat["period"] = strat["year"].apply(lambda y: "train" if y <= 2024 else "test")

CAPITAL_PER_TRADE = 1_000

strat["dollar_pnl"] = strat["strategy_pnl"] * CAPITAL_PER_TRADE

strat = strat.sort_values("exit_date").reset_index(drop=True)

strat["cum_dollar_pnl"]  = strat.groupby("period")["dollar_pnl"].cumsum()
strat["overall_cum_pnl"] = strat["dollar_pnl"].cumsum()
strat["total_invested"]  = (strat.index + 1) * CAPITAL_PER_TRADE
strat["roi"]             = strat["overall_cum_pnl"] / strat["total_invested"]
strat["peak"]            = strat["overall_cum_pnl"].cummax()
strat["drawdown"]        = strat["overall_cum_pnl"] - strat["peak"]

print(f"\nOverall Strategy Performance  (${CAPITAL_PER_TRADE:,} per trade)")
for period, grp in strat.groupby("period"):
    r   = grp["dollar_pnl"]
    inv = len(grp) * CAPITAL_PER_TRADE
    roi = r.sum() / inv
    print(f"\n  {period.upper()} ({grp['year'].min()}-{grp['year'].max()})")
    print(f"    Trades          : {len(r)}")
    print(f"    Capital deployed: ${inv:,.0f}")
    print(f"    Total PnL       : ${r.sum():,.2f}")
    print(f"    ROI             : {roi:.2%}")
    print(f"    Win rate        : {(r>0).mean():.1%}")
    print(f"    Avg per trade   : ${r.mean():,.2f}")
    print(f"    Best trade      : ${r.max():,.2f}")
    print(f"    Worst trade     : ${r.min():,.2f}")

r   = strat["dollar_pnl"]
inv = len(strat) * CAPITAL_PER_TRADE
roi = r.sum() / inv
print(f"\n  FULL PERIOD (2020-2025)")
print(f"    Trades          : {len(r)}")
print(f"    Capital deployed: ${inv:,.0f}")
print(f"    Total PnL       : ${r.sum():,.2f}")
print(f"    ROI             : {roi:.2%}")
print(f"    Win rate        : {(r>0).mean():.1%}")
print(f"    Avg per trade   : ${r.mean():,.2f}")
print(f"    Max Drawdown    : ${strat['drawdown'].min():,.2f}")
print(f"    Best trade      : ${r.max():,.2f}")
print(f"    Worst trade     : ${r.min():,.2f}")

fig = go.Figure()

for period, col, name in [("train", "#2E74B5", "Training 2020-2024"),
                           ("test",  GREEN,     "Test 2025")]:
    grp = strat[strat["period"]==period]
    fig.add_trace(go.Scatter(
        x=grp["exit_date"], y=grp["cum_dollar_pnl"],
        mode="lines", name=name,
        line=dict(color=col, width=2),
        hovertemplate="%{x|%b %Y}<br>Cumulative PnL: $%{y:,.0f}<extra></extra>",
    ))

fig.add_vline(x=pd.Timestamp("2025-01-01").timestamp()*1000,
              line_color=DG, line_dash="dash",
              annotation_text="2025 test begins",
              annotation_font=dict(family=FONT, size=11, color=GREY))
fig.add_hline(y=0, line_color=DG, line_dash="dot")

fig.update_layout(
    title=dict(text=f"Long/Short FOMC Straddle  ·  Cumulative PnL  (${CAPITAL_PER_TRADE:,} per trade)",
               font=dict(family=FONT, size=19, color="#111111"),
               x=0.03, xanchor="left"),
    paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(family=FONT, color="#111111", size=13),
    height=460, margin=dict(l=80, r=40, t=70, b=60),
    legend=dict(bgcolor=BG, bordercolor=DG, borderwidth=1,
                font=dict(family=FONT, size=12)),
    hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                    font=dict(family=FONT, color="#111111", size=13)),
    xaxis=dict(gridcolor=DG, linecolor=DG,
               tickfont=dict(family=FONT, color="#333333")),
    yaxis=dict(title="Cumulative PnL ($)",
               tickprefix="$", tickformat=",.0f",
               gridcolor=DG, linecolor=DG, zerolinecolor=DG,
               tickfont=dict(family=FONT, color="#333333"),
               title_font=dict(family=FONT, color="#333333", size=12)),
)
fig.show()


Overall Strategy Performance  ($1,000 per trade)

  TEST (2025-2025)
    Trades          : 263
    Capital deployed: $263,000
    Total PnL       : $28,439.36
    ROI             : 10.81%
    Win rate        : 74.1%
    Avg per trade   : $108.13
    Best trade      : $865.22
    Worst trade     : $-968.45

  TRAIN (2020-2024)
    Trades          : 1179
    Capital deployed: $1,179,000
    Total PnL       : $23,705.18
    ROI             : 2.01%
    Win rate        : 60.7%
    Avg per trade   : $20.11
    Best trade      : $1,184.66
    Worst trade     : $-1,454.04

  FULL PERIOD (2020-2025)
    Trades          : 1442
    Capital deployed: $1,442,000
    Total PnL       : $52,144.54
    ROI             : 3.62%
    Win rate        : 63.2%
    Avg per trade   : $36.16
    Max Drawdown    : $-44,560.49
    Best trade      : $1,184.66
    Worst trade     : $-1,454.04
